In [ ]:
## 带KV Cache版本的MHA
import torch
import torch.nn as nn
from einops import rearrange
import torch.nn.functional as F
from typing import Optional

class Attention(nn.Module):
    def __init__(self, args,hidden_size,num_heads):
        super().__init__()
        self.wq=nn.Linear(hidden_size,hidden_size)
        self.wk=nn.Linear(hidden_size,hidden_size)# 在这里改为group
        self.wv=nn.Linear(hidden_size,hidden_size)# 在这里改为group
        self.wo=nn.Linear(hidden_size,hidden_size)
        self.head_dim=hidden_size//num_heads# 头的维度
        # 初始化代码（省略与KV缓存无关的部分）
        self.cache_k = torch.zeros((args.max_batch_size, args.max_seq_len, self.n_local_heads, self.head_dim)).cuda()
        self.cache_v = torch.zeros((args.max_batch_size, args.max_seq_len, self.n_local_heads, self.head_dim)).cuda()
        # xq, xk = apply_rotary_emb(xq, xk, freqs_cis=freqs_cis)
    def apply_rotary_emb(xq, xk, freqs_cis=freqs_cis):
        return a,b

    def forward(self, x: torch.Tensor, start_pos: int, freqs_cis: torch.Tensor, mask: Optional[torch.Tensor]):
        bsz, seqlen, _ = x.shape
        xq, xk, xv = self.wq(x), self.wk(x), self.wv(x)
        
        # 使用einops进行张量重塑
        xq = rearrange(xq, 'b s (h d) -> b s h d', h=self.n_local_heads)
        xk = rearrange(xk, 'b s (h d) -> b s h d', h=self.n_local_heads)
        xv = rearrange(xv, 'b s (h d) -> b s h d', h=self.n_local_heads)
        
        xq, xk = apply_rotary_emb(xq, xk, freqs_cis=freqs_cis)
        
        self.cache_k = self.cache_k.to(xq)
        self.cache_v = self.cache_v.to(xq)
        
        # 更新缓存
        self.cache_k[:bsz, start_pos:start_pos+seqlen] = xk# 当前序列在缓存中的起始位置
        self.cache_v[:bsz, start_pos:start_pos+seqlen] = xv# 当前序列在缓存中的起始位置
        
        keys = self.cache_k[:bsz, :start_pos+seqlen]#读取到当前时间步的所有键值
        values = self.cache_v[:bsz, :start_pos+seqlen]# 读取到当前时间步的所有值
        
        # 使用einops进行矩阵乘法的维度排列
        xq = rearrange(xq, 'b s h d -> b h s d')
        keys = rearrange(keys, 'b s h d -> b h s d')
        values = rearrange(values, 'b s h d -> b h s d')
        scores = torch.matmul(xq, keys.transpose(-2, -1)) / torch.sqrt(self.head_dim)
        if mask is not None:
            scores = scores + mask
        scores = F.softmax(scores.float(), dim=-1).type_as(xq)
        output = torch.matmul(scores, values)
        output = rearrange(output, 'b h s d -> b s (h d)')
        return self.wo(output)

In [ ]:
import torch
from torch import nn
import einops
class MHA(nn.Module):
    def __init__(self,hidden_size,num_heads):
        super().__init__()
        self.q_l=nn.Linear(hidden_size,hidden_size)
        self.k_l=nn.Linear(hidden_size,hidden_size)# group
        self.v_l=nn.Linear(hidden_size,hidden_size)# group
        self.o_l=nn.Linear(hidden_size,hidden_size)
        self.head_dim=hidden_size//num_heads# 头的维度
    
    def forward(self, hs, mask=None):
        bz=hs.shape[0]# 第0维度
        q=self.q_l(hs)
        k=self.k_l(hs)
        v=self.v_l(hs)

        q=einops.rearange(q,"b seq_len (head head_dim) -> b head seq_len  head_dim")## d 代表每个头的维度
        k=einops.rearange(k,"b seq_len (head head_dim) -> b head seq_len  head_dim")## d 代表每个头的维度 # group
        v=einops.rearange(v,"b seq_len (head head_dim) -> b head seq_len  head_dim")## d 代表每个头的维度 # group

        attention_score=torch.matmul(q,k.transpose(-1,-2)/torch.sqrt(torch.tensor(self.head_dim)))
        # b head seq_len seq_len
        if mask !=None:
            attention_score=attention_score.masked_fill(mask==0,float("-inf"))
        # 归一化#对最后一维度的score进行归一化
        attention_prob=torch.softmax(attention_score,dim=-1)
        # seq_len seq_len @ seq_len head_dim
        out=torch.matmul(attention_prob,v)
        out=einops.rearrenge(out,"b head seq_len head_dim -> b seq_len (head head_dim)")
        out_final=self.o_l(out)
        return out_final
        

In [1]:
import torch
import torch.nn as nn
from einops import rearrange

class MQA(nn.Module):
    def __init__(self, hz, num_heads):
        super().__init__()
        self.q_l=nn.Linear(hz,hz)
        self.k_l=nn.Linear(hz,num_heads)
        self.v_l=nn.Linear(hz,num_heads)
        self.o_l=nn.Linear(hz,hz)
        self.head_dim=hz//num_heads

    def forward(self, hs, mask=None):
        q=self.q_l(hs)
        k=self.k_l(hs)
        v=self.v_l(hs)

        q=einops.rearrange(q,"b seq_len (head head_dim) -> b head seq_len head_dim")
        k=einops.rearrange(k,"b seq_len (head head_dim) -> b 1 seq_len head_dim")
        v=einops.rearrange(v,"b seq_len (head head_dim) -> b 1 seq_len head_dim")
        k=k.expand(-1,self.num_heads,-1,-1)
        v=v.expand(-1,self.num_heads,-1,-1)
        #  b h s d @ b 1 s d = b h s s
        attention_score=torch.malmul(q,k.transpose(-1,-2))/torch.sqrt(torch.tensor(self.head_dim))
        # if mask
        if mask !=None:
            attention_score=attention_score.masked_fill(mask==0,float("-inf"))
        attention_prob=torch.softmax(attention_score,dim=-1)
        out= torch.matmul(attention_prob,v)
        out=einops.rearrenge(out,"b head seq_len head_dim -> b seq_len (head head_dim)")
        out_final=self.o_l(out)
        return out_final

In [ ]:
import torch
from torch import nn
from einops import rearrange

class MHA(nn.Module):
    def __init__(self,hz,num_heads):
        super.__init__()
        self.head_dim=hz//num_heads
        self.q_l=nn.Linear(hz,hz)
        self.k_l=nn.Linear(hz,hz)
        self.v_l=nn.Linear(hz,hz)
        self.o_l=nn.Linear(hz,hz)

    def forward(self,hidden_state, mask=None):
        q=self.q_l(hidden_state)
        k=self.k_l(hidden_state)
        v=self.v_l(hidden_state)
        # o=self.o_l(hidden_state)

        q=einops.rearrange(q,"b seq_len (head head_dim) -> b head seq_len head_dim")
        k=einops.rearrange(k,"b seq_len (head head_dim) -> b head seq_len head_dim")
        v=einops.rearrange(v,"b seq_len (head head_dim) -> b head seq_len head_dim")

        attention_score=torch.malmut(q,k.transport(-1,-2))/torch.sqrt(self.head_dim)
        # b h s dim @ b h dim s = b h s s
        if mask==None:
            attention_score=attention_score.mask_filled(mask==0,float('-inf'))
        attention_prob=torch.softmax(attention_score,dim=-1)
        out=torch.matmul(attention_prob,v)
        out= einops.rearrenge(out,"b head seq_len head_dim -> b seq_len (head head_dim)")
        out= self.o_l(out)
        return out


In [ ]:
from typing import Optional, Tuple
import math
import torch
import torch.nn as nn
from einops import rearrange, einsum, repeat
from transformers.utils import logging

logger = logging.get_logger(__name__)


class MLA(nn.Module):
    """
    Multi-Head Latent Attention (MLA) with:
    1. 低秩瓶颈 (q_lora_rank / kv_lora_rank)
    2. 解耦 RoPE 位置编码
    3. 记忆高效的 einsum 注意力
    """

    def __init__(
        self,
        hidden_size: int = 2048,
        num_heads: int = 16,
        q_lora_rank: int = 128,          # Q 瓶颈
        kv_lora_rank: int = 512,         # KV 瓶颈
        qk_nope_head_dim: int = 128,     # 非 RoPE 部分
        qk_rope_head_dim: int = 64,      # RoPE 部分
        v_head_dim: int = 128,
        rope_theta: float = 10000.0,
        max_position_embeddings: int = 8192,
        attention_dropout: float = 0.0,
        layer_idx: Optional[int] = None,
    ):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.layer_idx = layer_idx
        self.q_lora_rank = q_lora_rank
        self.kv_lora_rank = kv_lora_rank

        # 每个头的维度
        self.q_head_dim = qk_nope_head_dim + qk_rope_head_dim
        self.k_head_dim = qk_nope_head_dim + qk_rope_head_dim
        self.v_head_dim = v_head_dim

        # ---------- 线性瓶颈 ---------- 先 down 在 up 记得中间加rope
        # Q 侧：h -> q_lora_rank -> num_heads * q_head_dim
        self.q_down = nn.Linear(hidden_size, q_lora_rank, bias=False)
        self.q_norm = nn.LayerNorm(q_lora_rank)
        self.q_up   = nn.Linear(q_lora_rank, num_heads * self.q_head_dim, bias=False)

        # KV 侧：h -> kv_lora_rank(+rope) -> num_heads * (k_head_dim + v_head_dim)
        self.kv_down = nn.Linear(
            hidden_size,
            kv_lora_rank + qk_rope_head_dim,  # 最后一维：kv_lora_rank | rope_dim
            bias=False,
        )
        self.kv_norm = nn.LayerNorm(kv_lora_rank)
        self.kv_up = nn.Linear(
            kv_lora_rank,
            num_heads * (qk_nope_head_dim + v_head_dim),
            bias=False,
        )

        # ---------- 输出 ----------
        self.out_proj = nn.Linear(num_heads * v_head_dim, hidden_size, bias=False)
        self.dropout = nn.Dropout(attention_dropout)

        # ---------- RoPE ----------
        self.register_buffer(
            "inv_freq",
            1.0 / (rope_theta ** (torch.arange(0, qk_rope_head_dim, 2).float() / qk_rope_head_dim)),
        )
        self.max_position_embeddings = max_position_embeddings

    # ------------- RoPE -------------
    def _get_cos_sin(self, seq_len, device, dtype):
        t = torch.arange(seq_len, device=device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        return emb.cos().to(dtype), emb.sin().to(dtype)

    @staticmethod
    def _rotate_half(x):
        x1, x2 = x.chunk(2, dim=-1)
        return torch.cat((-x2, x1), dim=-1)

    def _apply_rope(self, x, cos, sin, position_ids):
        """下面拆开讲一下，看完就明白 _rotate_half 的来龙去脉。
        复数视角：
        在 2-D 子空间里，一个向量 (x1, x2) 可看成复数 x1 + i x2。
        把它逆时针旋转角度 θ，就是乘 e^{iθ} = cosθ + i sinθ：
        于是旋转后的实部、虚部就是
        Copy
        x1′ =  x1 cosθ − x2 sinθ
        x2′ =  x1 sinθ + x2 cosθ"""
        # x: [bs, heads, seq_len, dim]
        cos = cos[position_ids].unsqueeze(1)  # [bs, 1, seq_len, dim]
        sin = sin[position_ids].unsqueeze(1)
        return x * cos + self._rotate_half(x) * sin

    # ------------- 前向 -------------
    def forward(
        self,
        hidden_states: torch.Tensor,          # [bs, seq_len, hidden_size]
        attention_mask: Optional[torch.Tensor] = None,  # [bs, seq_len]
        position_ids: Optional[torch.LongTensor] = None, #position_ids 就是每个 token 在序列中的绝对位置编号，
        past_key_value: Optional[Tuple[torch.Tensor]] = None,
        use_cache: bool = False,
    ) -> Tuple[torch.Tensor, ...]:
        bsz, q_len, _ = hidden_states.shape
        device, dtype = hidden_states.device, hidden_states.dtype

        # -------------------- Q 路径 --------------------
        q_latent = self.q_norm(self.q_down(hidden_states))  # [bs, q_len, q_lora_rank] 中间表征 先down 再up
        q_total  = rearrange(
            self.q_up(q_latent),
            "b l (h d) -> b h l d",
            h=self.num_heads,
        ) # [bs, heads, q_len, q_head_dim]
        q_nope, q_rope = q_total.split(
            [self.q_head_dim - self.qk_rope_head_dim, self.qk_rope_head_dim],
            dim=-1,
        ) # 分 头

        # -------------------- KV 路径 --------------------
        kv_mix = self.kv_down(hidden_states)                   # [bs, q_len, kv_lora_rank + rope_dim] 先down 再up
        kv_latent, k_rope = kv_mix.split(
            [self.kv_lora_rank, self.qk_rope_head_dim], dim=-1
        ) # 分出来 
        kv_latent = self.kv_norm(kv_latent) # [bs, q_len, kv_lora_rank]

        kv_up = rearrange(
            self.kv_up(kv_latent),
            "b l (h d) -> b h l d",
            h=self.num_heads,
        ) # [bs, heads, q_len, k_nope + v_head_dim]
        k_nope, value_states = kv_up.split(
            [self.q_head_dim - self.qk_rope_head_dim, self.v_head_dim],
            dim=-1,
        )
        k_rope = rearrange(k_rope, "b l d -> b 1 l d")# [bs, 1, q_len, rope_dim]

        # -------------------- RoPE --------------------
        kv_seq_len = q_len
        if past_key_value is not None:
            kv_seq_len += past_key_value[0].shape[-2]

        cos, sin = self._get_cos_sin(kv_seq_len, device, dtype)
        q_rope = self._apply_rope(q_rope, cos, sin, position_ids)
        k_rope = self._apply_rope(k_rope, cos, sin, position_ids)
        k_rope = repeat(k_rope, "b 1 l d -> b h l d", h=self.num_heads)

        # -------------------- 合并 --------------------
        query_states = torch.cat([q_nope, q_rope], dim=-1)     # [bs, heads, q_len, k_head_dim]
        key_states   = torch.cat([k_nope, k_rope], dim=-1)     # [bs, heads, q_len, k_head_dim]

        # -------------------- KV 缓存 --------------------
        if use_cache: # 缓存
            if past_key_value is None:
                past_key_value = (key_states, value_states)
            else: 
                key_states   = torch.cat([past_key_value[0], key_states],   dim=-2)
                value_states = torch.cat([past_key_value[1], value_states], dim=-2)
            present_key_value = (key_states, value_states)
        else:
            present_key_value = None

        # -------------------- 注意力 --------------------
        scores = einsum(query_states, key_states, "b h i d, b h j d -> b h i j")
        scores = scores / math.sqrt(self.q_head_dim)

        if attention_mask is not None:
            scores += attention_mask[:, None, None, :]  # 假设已预先构造 causal mask

        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        attn_out = einsum(attn_weights, value_states, "b h i j, b h j d -> b h i d")
        attn_out = rearrange(attn_out, "b h l d -> b l (h d)")  # [bs, q_len, num_heads * v_head_dim]

        output = self.out_proj(attn_out)                         # [bs, q_len, hidden_size]

        return output, attn_weights, present_key_value

```python
from einops import rearrenge
def compute_loss(self, input,group,rewards_func):
    rewards=rewards_func.sum(dim=1)# b g seq_len func
    
    mean_group_rewards=rearrenge(rewards,"(b g)")
    str_group_rewards=rearrenge()
```